In [1]:
pip install torch numpy tslearn pandas mantis-tsfm


   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/113.7 MB 3.7 MB/s eta 0:00:31
    --------------------------------------- 1.6/113.7 MB 4.0 MB/s eta 0:00:29
    --------------------------------------- 2.4/113.7 MB 3.7 MB/s eta 0:00:30
   - -------------------------------------- 3.4/113.7 MB 4.0 MB/s eta 0:00:28
   - -------------------------------------- 4.2/113.7 MB 3.9 MB/s eta 0:00:28
   - -------------------------------------- 5.2/113.7 MB 4.1 MB/s eta 0:00:27
   -- ------------------------------------- 6.0/113.7 MB 4.1 MB/s eta 0:00:27
   -- ------------------------------------- 6.8/113.7 MB 4.1 MB/s eta 0:00:27
   -- ------------------------------------- 7.9/113.7 MB 4.1 MB/s eta 0:00:26
   --- ------------------------------------ 8.7/113.7 MB 4.1 MB/s eta 0:00:26
   --- ------------------------------------ 9.4/113.7 MB 4.1 MB/s eta 0:00:26
   --- ------------------------------------ 10.2/113.7 MB 4.0 MB/s eta 

In [4]:
from sklearn.metrics import accuracy_score
#from tslearn.shapelets import LearningShapelets
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tslearn.datasets import UCR_UEA_datasets

In [5]:
from tslearn.datasets import UCR_UEA_datasets
# Load the LSST dataset from UEA archive
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

In [6]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(len(set(y_train)))

(2459, 36, 6)
(2466, 36, 6)
(2459,)
14


In [7]:
# normalisation
scaler = TimeSeriesScalerMeanVariance()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# encode labels (IMPORTANT)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# conversion torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

## Baseline model CNN

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNBaseline(nn.Module):
    def __init__(self, n_channels=6, n_classes=14):
        super().__init__()
        
        self.conv1 = nn.Conv1d(in_channels=n_channels, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        # x: (batch, 36, 6) -> (batch, 6, 36)
        x = x.permute(0, 2, 1)
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        x = self.pool(x).squeeze(-1)   # (batch, 128)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

## Step 2 : Initialiser modèle + boucle entraînement


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = CNNBaseline(n_channels=6, n_classes=14).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

cpu


In [15]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    
    return total_loss / total, correct / total

## Step 3 : Boucle d'évaluation

In [16]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    
    return total_loss / total, correct / total

## Lancer entraînement

In [22]:
n_epochs = 100

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1:02d}/{n_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Epoch 01/100 | Train Loss: 1.1516 | Train Acc: 0.6381 | Test Loss: 1.1851 | Test Acc: 0.6371
Epoch 02/100 | Train Loss: 1.1478 | Train Acc: 0.6299 | Test Loss: 1.1843 | Test Acc: 0.6375
Epoch 03/100 | Train Loss: 1.1313 | Train Acc: 0.6328 | Test Loss: 1.1833 | Test Acc: 0.6363
Epoch 04/100 | Train Loss: 1.1453 | Train Acc: 0.6356 | Test Loss: 1.1846 | Test Acc: 0.6383
Epoch 05/100 | Train Loss: 1.1387 | Train Acc: 0.6352 | Test Loss: 1.1839 | Test Acc: 0.6354
Epoch 06/100 | Train Loss: 1.1417 | Train Acc: 0.6336 | Test Loss: 1.1848 | Test Acc: 0.6363
Epoch 07/100 | Train Loss: 1.1447 | Train Acc: 0.6328 | Test Loss: 1.1840 | Test Acc: 0.6346
Epoch 08/100 | Train Loss: 1.1420 | Train Acc: 0.6393 | Test Loss: 1.1842 | Test Acc: 0.6371
Epoch 09/100 | Train Loss: 1.1441 | Train Acc: 0.6267 | Test Loss: 1.1834 | Test Acc: 0.6342
Epoch 10/100 | Train Loss: 1.1479 | Train Acc: 0.6307 | Test Loss: 1.1848 | Test Acc: 0.6342
Epoch 11/100 | Train Loss: 1.1467 | Train Acc: 0.6299 | Test Loss: 1.1

# Strong competitor (Setting 1) : Transformer model

In [19]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]


class TimeSeriesTransformerClassifier(nn.Module):
    def __init__(
        self,
        input_dim=6,
        seq_len=36,
        d_model=64,
        nhead=4,
        num_layers=3,
        dim_feedforward=128,
        dropout=0.1,
        n_classes=14
    ):
        super().__init__()

        self.input_projection = nn.Linear(input_dim, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len=seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # x: (batch, 36, 6)
        x = self.input_projection(x)              # (batch, 36, d_model)
        x = self.positional_encoding(x)           # (batch, 36, d_model)
        x = self.transformer(x)                   # (batch, 36, d_model)
        x = x.mean(dim=1)                         # global average pooling over time
        x = self.dropout(x)
        x = self.classifier(x)                    # (batch, n_classes)
        return x

## Step 2: Initialisation modèle

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

transformer_model = TimeSeriesTransformerClassifier(
    input_dim=6,
    seq_len=36,
    d_model=64,
    nhead=4,
    num_layers=3,
    dim_feedforward=128,
    dropout=0.1,
    n_classes=14
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=1e-4)

cpu


## Lancement entraînement

In [23]:
n_epochs = 100

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(
        transformer_model, train_loader, criterion, optimizer, device
    )
    test_loss, test_acc = evaluate(
        transformer_model, test_loader, criterion, device
    )

    print(
        f"Epoch {epoch+1:02d}/{n_epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
    )

Epoch 01/100 | Train Loss: 1.2471 | Train Acc: 0.5913 | Test Loss: 1.3000 | Test Acc: 0.5815
Epoch 02/100 | Train Loss: 1.2325 | Train Acc: 0.6035 | Test Loss: 1.3023 | Test Acc: 0.5843
Epoch 03/100 | Train Loss: 1.2246 | Train Acc: 0.5937 | Test Loss: 1.2943 | Test Acc: 0.5766
Epoch 04/100 | Train Loss: 1.2126 | Train Acc: 0.6002 | Test Loss: 1.2869 | Test Acc: 0.5896
Epoch 05/100 | Train Loss: 1.2072 | Train Acc: 0.5986 | Test Loss: 1.2791 | Test Acc: 0.5884
Epoch 06/100 | Train Loss: 1.1946 | Train Acc: 0.6059 | Test Loss: 1.2747 | Test Acc: 0.5908
Epoch 07/100 | Train Loss: 1.1821 | Train Acc: 0.6080 | Test Loss: 1.2833 | Test Acc: 0.5880
Epoch 08/100 | Train Loss: 1.1955 | Train Acc: 0.5986 | Test Loss: 1.2754 | Test Acc: 0.5929
Epoch 09/100 | Train Loss: 1.1780 | Train Acc: 0.6169 | Test Loss: 1.2694 | Test Acc: 0.5945
Epoch 10/100 | Train Loss: 1.1741 | Train Acc: 0.6141 | Test Loss: 1.2766 | Test Acc: 0.5864
Epoch 11/100 | Train Loss: 1.1550 | Train Acc: 0.6198 | Test Loss: 1.2